# Beyond Jev: Trying Five Decision Models

One section per model: get access, paste your key, make a request, inspect the answer, and time it.

Run the shared setup below once. Then jump to any model section you have access to; you do not need all five accounts.

1. [Perplexity Decider](#1.-Perplexity-Decider)
2. [OpenAI Decisions](#2.-OpenAI-Decisions)
3. [Cloudflare Clef](#3.-Cloudflare-Clef)
4. [Liquid d1-ultra](#4.-Liquid-d1-ultra)
5. [Fastino GLiDE](#5.-Fastino-GLiDE)

> **Demo keys:** The setup cells deliberately accept API keys directly in this notebook. Keep them off camera, and remove them and clear outputs before committing or sharing the file. Use environment variables or a secret manager in real projects. Revoke a key if it is exposed.

These are live, billed API calls. Outputs are intentionally empty in the shared notebook; fill in your keys to generate your own results. API examples were checked against the linked vendor docs on 2026-10-25.


## Shared setup

From this folder, run `uv sync`, then open this notebook with `nvb decision_models.ipynb` or `uv run jupyter lab decision_models.ipynb`.

All five sections use plain HTTP so the endpoint, request body, and response differences stay visible. Timing measures the client round trip, including network and service queueing; it is not isolated model inference.


In [ ]:
import httpx
from time import perf_counter
from statistics import median
from IPython.display import Markdown, display

# Holds only measurements from sections you actually run.
timings = {}


### The same questions for every model

Start with a browser-specific bug and a workaround. Then try an outage affecting everyone. We ask for a team, a yes/no urgency probability, and a severity rating.

The three severity levels are indexed 0, 1, and 2. Returned probabilities are estimates; a large number is not proof of calibration on your data.


In [ ]:
EXAMPLE = (
    "CSV export fails in Safari but works in Chrome. "
    "I can still finish my work in Chrome."
)
CHANGED_EXAMPLE = (
    "Checkout is failing for every customer in every browser. "
    "No purchases have succeeded for the last hour, and there is no workaround."
)

TEAM_QUESTION = "Which team should handle this report?"
TEAM_OPTIONS = {
    "billing": "Charges, invoices, refunds, and payment methods",
    "technical": "Software bugs, outages, errors, and browser problems",
    "account": "Login, account access, profile, and permissions",
}
URGENT_QUESTION = (
    "Does this require immediate incident response? "
    "A widespread outage with no workaround is urgent; "
    "a browser-specific problem with a working alternative is not."
)
SEVERITY_QUESTION = "How severe is the reported impact?"
SEVERITY_LEVELS = [
    "Cosmetic: appearance only; functionality still works",
    "Workaround available: a task fails but another way works",
    "Blocked: a core task fails with no workaround",
]

print(EXAMPLE)


## 1. Perplexity Decider

[Get an API key](https://console.perplexity.ai/project/keys) · [Decisions docs](https://docs.perplexity.ai/docs/decisions/quickstart)

Create a Perplexity API key, enable billing or credits if required, then paste it below.

Uses `state` and named questions. `choice`, `noul`, and `score` answers are returned in an `answers` object. The Flash sibling can be explored by explicitly changing the model to `pplx-decider-flash-v1.2`; it is a different model with different multi-question billing.


In [ ]:
PERPLEXITY_API_KEY = ""  # Paste your demo key here

if not PERPLEXITY_API_KEY.strip():
    raise ValueError("Paste your PERPLEXITY_API_KEY into this section’s setup cell.")

perplexity_model = "pplx-decider-v1.2"
perplexity_client = httpx.Client(
    headers={"Authorization": f"Bearer {PERPLEXITY_API_KEY}"},
    timeout=60.0,
)


In [ ]:
def ask_perplexity(text):
    payload = {
        "model": perplexity_model,
        "state": text,
        "questions": {
            "team": {
                "type": "choice",
                "instructions": TEAM_QUESTION,
                "criteria": TEAM_OPTIONS,
            },
            "urgent": {"type": "noul", "instructions": URGENT_QUESTION},
            "severity": {
                "type": "score",
                "instructions": SEVERITY_QUESTION,
                "criteria": SEVERITY_LEVELS,
            },
        },
    }
    start = perf_counter()
    response = perplexity_client.post(
        "https://api.perplexity.ai/v1/decisions",
        json=payload,
    )
    elapsed_ms = (perf_counter() - start) * 1_000
    response.raise_for_status()
    return response.json(), elapsed_ms


In [ ]:
perplexity_raw, perplexity_first_ms = ask_perplexity(EXAMPLE)
perplexity_result = perplexity_raw
timings["Perplexity Decider"] = {"model": perplexity_model, "first_ms": perplexity_first_ms}
print(f"Perplexity Decider: {perplexity_first_ms:.0f} ms")
perplexity_result["answers"]


Compare the full distributions. Perplexity documents `confidence` separately from the highest option probability; they are not the same number.

### Change the input

The first example has a workaround; the second is a widespread outage. Look at how the urgency probability and severity distribution change.


In [ ]:
perplexity_changed_raw, perplexity_changed_ms = ask_perplexity(CHANGED_EXAMPLE)
perplexity_changed = perplexity_changed_raw
print(f"Changed input: {perplexity_changed_ms:.0f} ms")
perplexity_changed["answers"]


### Optional: three more timings

Run the original text three more times with the same client and questions. This makes three additional billed requests. These are repeated, warm-client measurements; caching and provider scheduling can affect them.


In [ ]:
perplexity_repeated_ms = []
for _ in range(3):
    perplexity_repeat_raw, elapsed_ms = ask_perplexity(EXAMPLE)
    perplexity_repeated_ms.append(elapsed_ms)

timings["Perplexity Decider"]["repeated_median_ms"] = median(perplexity_repeated_ms)
print("Repeated calls (ms):", [round(ms) for ms in perplexity_repeated_ms])
print(f"Median: {median(perplexity_repeated_ms):.0f} ms")


## 2. OpenAI Decisions

[Get an API key](https://platform.openai.com/api-keys) · [Decisions docs](https://developers.openai.com/api/docs/guides/decisions)

Create an OpenAI API key with API billing and Decisions access, then paste it below. A ChatGPT subscription does not supply this key.

Uses `input` and an array of named questions. The yes/no type is called `predicate`, and answers come back as an array. This is the Decisions endpoint, not a chat-completions request asking for JSON.


In [ ]:
OPENAI_API_KEY = ""  # Paste your demo key here

if not OPENAI_API_KEY.strip():
    raise ValueError("Paste your OPENAI_API_KEY into this section’s setup cell.")

openai_model = "gpt-6-luna"
openai_client = httpx.Client(
    headers={"Authorization": f"Bearer {OPENAI_API_KEY}"},
    timeout=60.0,
)


In [ ]:
def ask_openai(text):
    payload = {
        "model": openai_model,
        "input": text,
        "questions": [
            {
                "type": "choice", "name": "team",
                "instructions": TEAM_QUESTION,
                "choices": [
                    {"value": name, "description": description}
                    for name, description in TEAM_OPTIONS.items()
                ],
            },
            {"type": "predicate", "name": "urgent", "instructions": URGENT_QUESTION},
            {
                "type": "score", "name": "severity",
                "instructions": SEVERITY_QUESTION,
                "levels": [{"label": level, "description": level} for level in SEVERITY_LEVELS],
            },
        ],
    }
    start = perf_counter()
    response = openai_client.post(
        "https://api.openai.com/v1/decisions",
        json=payload,
    )
    elapsed_ms = (perf_counter() - start) * 1_000
    response.raise_for_status()
    return response.json(), elapsed_ms


In [ ]:
openai_raw, openai_first_ms = ask_openai(EXAMPLE)
openai_result = openai_raw
timings["OpenAI Decisions"] = {"model": openai_model, "first_ms": openai_first_ms}
print(f"OpenAI Decisions: {openai_first_ms:.0f} ms")
openai_result["answers"]


Inspect `answers` before assuming every item contains a decision: the API can return a refusal. The response can echo the requested model name rather than prove an immutable checkpoint.

### Change the input

The first example has a workaround; the second is a widespread outage. Look at how the urgency probability and severity distribution change.


In [ ]:
openai_changed_raw, openai_changed_ms = ask_openai(CHANGED_EXAMPLE)
openai_changed = openai_changed_raw
print(f"Changed input: {openai_changed_ms:.0f} ms")
openai_changed["answers"]


### Optional: three more timings

Run the original text three more times with the same client and questions. This makes three additional billed requests. These are repeated, warm-client measurements; caching and provider scheduling can affect them.


In [ ]:
openai_repeated_ms = []
for _ in range(3):
    openai_repeat_raw, elapsed_ms = ask_openai(EXAMPLE)
    openai_repeated_ms.append(elapsed_ms)

timings["OpenAI Decisions"]["repeated_median_ms"] = median(openai_repeated_ms)
print("Repeated calls (ms):", [round(ms) for ms in openai_repeated_ms])
print(f"Median: {median(openai_repeated_ms):.0f} ms")


## 3. Cloudflare Clef

[Workers AI dashboard](https://dash.cloudflare.com/?to=/:account/ai/workers-ai) · [Token and account-ID setup](https://developers.cloudflare.com/workers-ai/get-started/rest-api/) · [Clef docs](https://developers.cloudflare.com/workers-ai/models/clef/)

In Workers AI, choose **Use REST API**, create the Workers AI API token, and copy your **Account ID**. Paste both below. This notebook calls the REST endpoint directly; no Worker deployment is needed.

The model runs on Workers AI. The REST response wraps the model output in `result`, alongside `success` and `errors`. This section selects Clef explicitly; Clef-flash has a different endpoint and model selector.


In [ ]:
CLOUDFLARE_API_TOKEN = ""  # Paste your demo key here
CLOUDFLARE_ACCOUNT_ID = ""  # Copy from Workers AI → Use REST API

if not CLOUDFLARE_API_TOKEN.strip():
    raise ValueError("Paste your CLOUDFLARE_API_TOKEN into this section’s setup cell.")
if not CLOUDFLARE_ACCOUNT_ID.strip():
    raise ValueError("Paste your Cloudflare Account ID into this setup cell.")

cloudflare_model = "clef"
cloudflare_client = httpx.Client(
    headers={"Authorization": f"Bearer {CLOUDFLARE_API_TOKEN}"},
    timeout=60.0,
)


In [ ]:
def ask_cloudflare(text):
    payload = {
        "model": cloudflare_model,
        "state": text,
        "questions": {
            "team": {
                "type": "choice",
                "instructions": TEAM_QUESTION,
                "criteria": TEAM_OPTIONS,
            },
            "urgent": {"type": "noul", "instructions": URGENT_QUESTION},
            "severity": {
                "type": "score",
                "instructions": SEVERITY_QUESTION,
                "criteria": SEVERITY_LEVELS,
            },
        },
    }
    start = perf_counter()
    response = cloudflare_client.post(
        f"https://api.cloudflare.com/client/v4/accounts/{CLOUDFLARE_ACCOUNT_ID}/ai/run/@cf/cloudflare/clef",
        json=payload,
    )
    elapsed_ms = (perf_counter() - start) * 1_000
    response.raise_for_status()
    return response.json(), elapsed_ms


In [ ]:
cloudflare_raw, cloudflare_first_ms = ask_cloudflare(EXAMPLE)
if not cloudflare_raw["success"]:
    raise RuntimeError(cloudflare_raw["errors"])
cloudflare_result = cloudflare_raw["result"]
timings["Cloudflare Clef"] = {"model": cloudflare_model, "first_ms": cloudflare_first_ms}
print(f"Cloudflare Clef: {cloudflare_first_ms:.0f} ms")
cloudflare_result["answers"]


Cloudflare wraps the model response, so inspect `cloudflare_raw` to see the complete HTTP JSON, or `cloudflare_result` for the decision payload.

### Change the input

The first example has a workaround; the second is a widespread outage. Look at how the urgency probability and severity distribution change.


In [ ]:
cloudflare_changed_raw, cloudflare_changed_ms = ask_cloudflare(CHANGED_EXAMPLE)
if not cloudflare_changed_raw["success"]:
    raise RuntimeError(cloudflare_changed_raw["errors"])
cloudflare_changed = cloudflare_changed_raw["result"]
print(f"Changed input: {cloudflare_changed_ms:.0f} ms")
cloudflare_changed["answers"]


### Optional: three more timings

Run the original text three more times with the same client and questions. This makes three additional billed requests. These are repeated, warm-client measurements; caching and provider scheduling can affect them.


In [ ]:
cloudflare_repeated_ms = []
for _ in range(3):
    cloudflare_repeat_raw, elapsed_ms = ask_cloudflare(EXAMPLE)
    if not cloudflare_repeat_raw["success"]:
        raise RuntimeError(cloudflare_repeat_raw["errors"])
    cloudflare_repeated_ms.append(elapsed_ms)

timings["Cloudflare Clef"]["repeated_median_ms"] = median(cloudflare_repeated_ms)
print("Repeated calls (ms):", [round(ms) for ms in cloudflare_repeated_ms])
print(f"Median: {median(cloudflare_repeated_ms):.0f} ms")


## 4. Liquid d1-ultra

[Get an API key](https://console.liquid.ai) · [Hosted d1 docs](https://docs.liquid.ai/lfm/models/d1) · [Open d1 models](https://docs.liquid.ai/lfm/models/decision-models)

In the Liquid Console, join an organization and open **Dashboard → API Keys**. Create a key, enable credits if required, and paste it below.

This calls the hosted `d1-ultra` model. It does not download or run `d1-3B` locally. Liquid documents `choice`, `noul`, and `score` on its System One-format endpoint.


In [ ]:
LIQUID_API_KEY = ""  # Paste your demo key here

if not LIQUID_API_KEY.strip():
    raise ValueError("Paste your LIQUID_API_KEY into this section’s setup cell.")

liquid_model = "d1-ultra"
liquid_client = httpx.Client(
    headers={"Authorization": f"Bearer {LIQUID_API_KEY}"},
    timeout=60.0,
)


In [ ]:
def ask_liquid(text):
    payload = {
        "model": liquid_model,
        "state": text,
        "questions": {
            "team": {
                "type": "choice",
                "instructions": TEAM_QUESTION,
                "criteria": TEAM_OPTIONS,
            },
            "urgent": {"type": "noul", "instructions": URGENT_QUESTION},
            "severity": {
                "type": "score",
                "instructions": SEVERITY_QUESTION,
                "criteria": SEVERITY_LEVELS,
            },
        },
    }
    start = perf_counter()
    response = liquid_client.post(
        "https://api.liquid.ai/decisions/v1/systemone",
        json=payload,
    )
    elapsed_ms = (perf_counter() - start) * 1_000
    response.raise_for_status()
    return response.json(), elapsed_ms


In [ ]:
liquid_raw, liquid_first_ms = ask_liquid(EXAMPLE)
liquid_result = liquid_raw
timings["Liquid d1-ultra"] = {"model": liquid_model, "first_ms": liquid_first_ms}
print(f"Liquid d1-ultra: {liquid_first_ms:.0f} ms")
liquid_result["answers"]


Liquid bills the shared text again for each question. Keep the question count fixed when comparing these round trips. Hosted d1-ultra and open d1-3B are separate models.

### Change the input

The first example has a workaround; the second is a widespread outage. Look at how the urgency probability and severity distribution change.


In [ ]:
liquid_changed_raw, liquid_changed_ms = ask_liquid(CHANGED_EXAMPLE)
liquid_changed = liquid_changed_raw
print(f"Changed input: {liquid_changed_ms:.0f} ms")
liquid_changed["answers"]


### Optional: three more timings

Run the original text three more times with the same client and questions. This makes three additional billed requests. These are repeated, warm-client measurements; caching and provider scheduling can affect them.


In [ ]:
liquid_repeated_ms = []
for _ in range(3):
    liquid_repeat_raw, elapsed_ms = ask_liquid(EXAMPLE)
    liquid_repeated_ms.append(elapsed_ms)

timings["Liquid d1-ultra"]["repeated_median_ms"] = median(liquid_repeated_ms)
print("Repeated calls (ms):", [round(ms) for ms in liquid_repeated_ms])
print(f"Median: {median(liquid_repeated_ms):.0f} ms")


## 5. Fastino GLiDE

[Get an API key](https://agent.fastino.ai/api-keys) · [GLiDE API docs](https://docs.fastino.ai/inference/systemone) · [Model card](https://huggingface.co/fastino/GLiDE)

Create a Fastino account, open **Settings → API keys**, and paste your key below. Make sure the account has credits and access to GLiDE.

Uses `X-API-Key` authentication and the exact selector `fastino/GLiDE`. The timeout follows the hosted API docs. The request makes storage behavior explicit with `store=False`.


In [ ]:
FASTINO_API_KEY = ""  # Paste your demo key here

if not FASTINO_API_KEY.strip():
    raise ValueError("Paste your FASTINO_API_KEY into this section’s setup cell.")

fastino_model = "fastino/GLiDE"
fastino_client = httpx.Client(
    headers={"X-API-Key": FASTINO_API_KEY},
    timeout=300.0,
)


In [ ]:
def ask_fastino(text):
    payload = {
        "model": fastino_model,
        "state": text,
        "questions": {
            "team": {
                "type": "choice",
                "instructions": TEAM_QUESTION,
                "criteria": TEAM_OPTIONS,
            },
            "urgent": {"type": "noul", "instructions": URGENT_QUESTION},
            "severity": {
                "type": "score",
                "instructions": SEVERITY_QUESTION,
                "criteria": SEVERITY_LEVELS,
            },
        },
        "store": False,
    }
    start = perf_counter()
    response = fastino_client.post(
        "https://api.fastino.ai/v1/systemone",
        json=payload,
    )
    elapsed_ms = (perf_counter() - start) * 1_000
    response.raise_for_status()
    return response.json(), elapsed_ms


In [ ]:
fastino_raw, fastino_first_ms = ask_fastino(EXAMPLE)
fastino_result = fastino_raw
timings["Fastino GLiDE"] = {"model": fastino_model, "first_ms": fastino_first_ms}
print(f"Fastino GLiDE: {fastino_first_ms:.0f} ms")
fastino_result["answers"]


GLiDE uses `score` for the winning level index and `expected_level` for the probability-weighted score. Compare `expected_level` with the other providers’ weighted scores. A cold deployment can return HTTP 425; inspect the error and rerun after the documented delay. This notebook does not hide retries inside its timing.

### Change the input

The first example has a workaround; the second is a widespread outage. Look at how the urgency probability and severity distribution change.


In [ ]:
fastino_changed_raw, fastino_changed_ms = ask_fastino(CHANGED_EXAMPLE)
fastino_changed = fastino_changed_raw
print(f"Changed input: {fastino_changed_ms:.0f} ms")
fastino_changed["answers"]


### Optional: three more timings

Run the original text three more times with the same client and questions. This makes three additional billed requests. These are repeated, warm-client measurements; caching and provider scheduling can affect them.


In [ ]:
fastino_repeated_ms = []
for _ in range(3):
    fastino_repeat_raw, elapsed_ms = ask_fastino(EXAMPLE)
    fastino_repeated_ms.append(elapsed_ms)

timings["Fastino GLiDE"]["repeated_median_ms"] = median(fastino_repeated_ms)
print("Repeated calls (ms):", [round(ms) for ms in fastino_repeated_ms])
print(f"Median: {median(fastino_repeated_ms):.0f} ms")


## Compare the timings you actually ran

This table includes only initialized sections whose first request succeeded. Run the optional timing cells to add repeated-call medians. Every entry uses the original text and the same three question meanings; providers can still evaluate and bill those questions differently.

These handfuls of calls illustrate your account, network, and serving conditions. Compare the answers as well as the speed. The provider-specific `confidence` fields are not interchangeable, and a fast incorrect answer is not a win.


In [ ]:
rows = ["| Model | First call (ms) | Repeated median (ms) |", "|---|---:|---:|"]
for name, result in timings.items():
    repeated = (
        f"{result['repeated_median_ms']:.0f}"
        if "repeated_median_ms" in result else "Not run"
    )
    rows.append(f"| {name} (`{result['model']}`) | {result['first_ms']:.0f} | {repeated} |")
display(Markdown("\n".join(rows)))


## Try your own example

Edit `EXAMPLE`, rerun the shared questions cell, and rerun each model's first-call and timing cells. If you change the example, clear `timings` by rerunning shared setup before collecting a new comparison.

For the full HTTP response, inspect a section's `*_raw` variable. Versioned names and aliases can move; retain the raw `model` value when recording a comparison. If you change the model selector, rerun that model's first-call cell before its timing cell.

When finished, close the HTTP clients in the sections you ran. Remove your API keys and clear outputs before sharing the notebook.
